# F5-TTS narration on a free Kaggle GPU

Generates one audio file per script segment in **your cloned voice**, then zips them.

**Setup (once):**
1. Kaggle → Create → New Notebook. In the right panel set **Accelerator = GPU T4 x2** (or P100).
2. **Add Input → Upload** the `*_narration_kit.zip` from the pipeline as a Dataset (it contains `manifest.json` + `reference.wav`).
3. Run all cells. Download `narration_audio.zip` from the output, send it back, and assemble with `provider: prerecorded`.

In [ ]:
!pip -q install f5-tts soundfile
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import glob, json, os
# Find the uploaded kit (manifest.json + reference.wav) under /kaggle/input/.
manifest_path = next(iter(glob.glob('/kaggle/input/**/manifest.json', recursive=True)), None)
assert manifest_path, 'Upload the *_narration_kit.zip as a Dataset (Add Input).'
kit_dir = os.path.dirname(manifest_path)
manifest = json.load(open(manifest_path))
ref_audio = os.path.join(kit_dir, 'reference.wav')
ref_text = manifest.get('ref_text', '') or ''
print('slug:', manifest['slug'], '| segments:', len(manifest['segments']))

In [ ]:
from f5_tts.api import F5TTS
tts = F5TTS(model='F5TTS_v1_Base')  # downloads weights once
os.makedirs('/kaggle/working/narration', exist_ok=True)
for seg in manifest['segments']:
    out = f"/kaggle/working/narration/{seg['segment_id']}.wav"
    tts.infer(ref_file=ref_audio, ref_text=ref_text, gen_text=seg['text'],
              file_wave=out, nfe_step=32, remove_silence=True)
    print('done', seg['segment_id'])

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/narration_audio', 'zip', '/kaggle/working/narration')
print('Wrote /kaggle/working/narration_audio.zip — download it from the Output panel.')